### Answering the buisiness question:
**How can we optimize 28-day safety stock levels and reorder points for store CA_1 to minimize inventory holding costs while preventing stockouts on fast-moving household and food items?**


for faster computation time, we would stick with top 20 or 30% SKU s from the CA_1 store

In [1]:
import sys
from pathlib import Path

# Adds the root_dir (parent of notebooks/) to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


BASE_DIR = Path.cwd().parent
BASE_DIR

DATA_DIR = BASE_DIR/"data"/"processed"

parent_dir

'/media/ashfaque/datas/ML-projects/retail-forecast-system'

In [2]:
# all libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from datetime import datetime
import joblib
import lightgbm as lgb 
plt.style.use('seaborn-v0_8-darkgrid')

from src.metrics import wrmsse

# from train_and_eval import *

import warnings


In [3]:
# load the data

sales_ca1 = pd.read_parquet(DATA_DIR/"sales_known_ca_1.parquet")
sales_future_ca1 = pd.read_parquet(DATA_DIR/"sales_future_ca_1.parquet")

print(sales_ca1.shape, sales_future_ca1.shape)
print(" number of training days: ",(sales_ca1['date'].max()-sales_ca1['date'].min()).days)
print(" number of future unknown days: ", (sales_future_ca1['date'].max()-sales_future_ca1['date'].min()).days)
print(" maximum training date: ", sales_ca1['date'].max())
print(" number of unique items in store: ",len(sales_ca1['item_id'].unique()))

(4367553, 25) (420714, 25)
 number of training days:  1802
 number of future unknown days:  137
 maximum training date:  2016-01-05 00:00:00
 number of unique items in store:  3047


In [4]:
sales_ca1 = sales_ca1[sales_ca1['sell_price'].notna()].copy()
sales_ca1['sell_price'].isna().sum()

np.int64(0)

### Filtering the items to top 20%, or 30% items

In [5]:
def get_items_top(df,percentile):
    df_temp = df.copy()

    df_temp['revenue'] = df_temp['sales']*df_temp['sell_price']
    #aggregate
    item_revenue = df_temp.groupby('item_id',observed=True)['revenue'].sum().reset_index()

    # calculate top 80th percentile
    top_20 = item_revenue['revenue'].quantile(percentile)
    top_20_items = item_revenue[item_revenue['revenue'] >= top_20]['item_id'].tolist()

    df_temp_top = df_temp[df_temp['item_id'].isin(top_20_items)].copy()
    df_temp_top['item_id'] = df_temp_top['item_id'].cat.remove_unused_categories()

    return df_temp_top

def get_items_with_full_history(df):
    '''returns items spanning the entire history of given dataframe''' 
    total_history = (df['date'].max()-df['date'].min()).days
    print("total history: ",total_history)

    item_stats = df.groupby('item_id',observed=True)['date'].agg(min_date='min',max_date='max',total_records='nunique').reset_index()
    item_stats['history_span'] = (item_stats['max_date'] - item_stats['min_date']).dt.days

    # 4. Create a boolean mask for items present across the full timeframe
    # Option A: Spans the full start-to-end range
    full_span_mask = item_stats['history_span'] == total_history

    valid_full_history_items = item_stats[full_span_mask]['item_id'].tolist()

    print(f"Total unique items: {len(item_stats)}")
    print(f"Items with full history: {len(valid_full_history_items)}")

    # 6. Filter your main DataFrame to keep only items with complete historical data
    df_complete = df[df['item_id'].isin(valid_full_history_items)].copy()
    return df_complete
   

In [6]:
items_top_20_perc = get_items_top(sales_ca1,0.8)
items_full_history = get_items_with_full_history(items_top_20_perc)

items_full_history['item_id'].unique()

total history:  1802
Total unique items: 610
Items with full history: 373


['FOODS_1_018', 'FOODS_1_024', 'FOODS_1_032', 'FOODS_1_044', 'FOODS_1_045', ..., 'HOUSEHOLD_2_450', 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']
Length: 373
Categories (610, object): ['FOODS_1_004', 'FOODS_1_012', 'FOODS_1_018', 'FOODS_1_023', ..., 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']

In [7]:
# fastmoving top 20%, 30% items in list for faster calculations

sales_ca1['revenue'] = sales_ca1['sales']*sales_ca1['sell_price']
# aggregate
item_revenue = sales_ca1.groupby('item_id',observed=True)['revenue'].sum().reset_index()

# calculate top 80th and 70th percentile 
top_20 = item_revenue['revenue'].quantile(0.8)
top_30 = item_revenue['revenue'].quantile(0.7)

top_20_items = item_revenue[item_revenue['revenue'] >= top_20]['item_id'].tolist()
top_30_items = item_revenue[item_revenue['revenue'] >= top_30]['item_id'].tolist()

sales_ca1_fast = sales_ca1[sales_ca1['item_id'].isin(top_20_items)].copy()
sales_ca1_fast['item_id'] = sales_ca1_fast['item_id'].cat.remove_unused_categories()


In [8]:
sales_ca1_fast['item_id'].unique()

['FOODS_1_004', 'FOODS_1_012', 'FOODS_1_018', 'FOODS_1_023', 'FOODS_1_024', ..., 'HOUSEHOLD_2_453', 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']
Length: 610
Categories (610, object): ['FOODS_1_004', 'FOODS_1_012', 'FOODS_1_018', 'FOODS_1_023', ..., 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']

* for  backtests, we further filter the items to having the entire history

In [9]:

# now pick items that spans the entire available history

total_history = (sales_ca1_fast['date'].max()-sales_ca1_fast['date'].min()).days
print("total history: ",total_history)

item_stats = sales_ca1_fast.groupby('item_id',observed=True)['date'].agg(min_date='min',max_date='max',total_records='nunique').reset_index()
item_stats['history_span'] = (item_stats['max_date'] - item_stats['min_date']).dt.days


# 4. Create a boolean mask for items present across the full timeframe
# Option A: Spans the full start-to-end range
full_span_mask = item_stats['history_span'] == total_history

valid_full_history_items = item_stats[full_span_mask]['item_id'].tolist()

print(f"Total unique items: {len(item_stats)}")
print(f"Items with full history: {len(valid_full_history_items)}")

# 6. Filter your main DataFrame to keep only items with complete historical data
sales_ca1_complete = sales_ca1_fast[sales_ca1_fast['item_id'].isin(valid_full_history_items)].copy()
sales_ca1_complete.head()

total history:  1802
Total unique items: 610
Items with full history: 373


/media/ashfaque/datas/ML-projects/retail-forecast-system/.venv/lib/python3.12/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/media/ashfaque/datas/ML-projects/retail-forecast-system/.venv/lib/python3.12/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day_of_week,day_of_month,week_of_year,revenue
25866,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_1,17,2011-01-29,11101,...,NaN,NaN,0,0,0,1.0,5,29,4,17.0
25867,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_2,3,2011-01-30,11101,...,NaN,NaN,0,0,0,1.0,6,30,4,3.0
25868,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_3,8,2011-01-31,11101,...,NaN,NaN,0,0,0,1.0,0,31,5,8.0
25869,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_4,2,2011-02-01,11101,...,NaN,NaN,1,1,0,1.0,1,1,5,2.0
25870,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_5,5,2011-02-02,11101,...,NaN,NaN,1,0,1,1.0,2,2,5,5.0


##  Backtesting experiment:

* model = lightgbm
* forecasting method = recursive
* horizon = 28
* history = full
* metric  = wrmsse 
* items   = 373, spanning entire history

### Walk forward expanding window

In [7]:
from src.pipeline import recursive_forecast_batch
from src.backtests import walk_forward_expanding_window
from train_and_eval import *

train_data = sales_ca1_complete.copy()
# cutoff = train_data['date'].max()-pd.Timedelta(days=28)

# train = train_data[train_data['date']<=cutoff]
# test  = train_data[train_data['date']>cutoff]

# train_df = build_train_features(train)
# models, cat_categories, params = train_models(train_df)

# pred_df = predict_eval_set(models,train_df,test,cat_categories)

# print(f"computing WRMSSE")
# score = wrmsse(train_df,test,pred_df)
# print(f" WRMSSE: {score:.4f}")

In [8]:
wndw_dates, metrics, models, preds = walk_forward_expanding_window(train_data)

testing window:1
Training period: 2011-01-29 00:00:00-2012-01-29 00:00:00, WRMSSE:   0.7972116753997809
testing window:2
Training period: 2011-01-29 00:00:00-2012-06-17 00:00:00, WRMSSE:   0.9015404019115072
testing window:3
Training period: 2011-01-29 00:00:00-2012-11-04 00:00:00, WRMSSE:   0.7687436223025507
testing window:4
Training period: 2011-01-29 00:00:00-2013-03-24 00:00:00, WRMSSE:   0.8047169751211585
testing window:5
Training period: 2011-01-29 00:00:00-2013-08-11 00:00:00, WRMSSE:   0.8919754437164895
testing window:6
Training period: 2011-01-29 00:00:00-2013-12-29 00:00:00, WRMSSE:   0.7795234987111022
testing window:7
Training period: 2011-01-29 00:00:00-2014-05-18 00:00:00, WRMSSE:   0.8011349149082363
testing window:8
Training period: 2011-01-29 00:00:00-2014-10-05 00:00:00, WRMSSE:   0.7844716887661726
testing window:9
Training period: 2011-01-29 00:00:00-2015-02-22 00:00:00, WRMSSE:   0.7479558684011509
testing window:10
Training period: 2011-01-29 00:00:00-2015-07-1

In [9]:
metrics

{'window_1': 0.7972116753997809,
 'window_2': 0.9015404019115072,
 'window_3': 0.7687436223025507,
 'window_4': 0.8047169751211585,
 'window_5': 0.8919754437164895,
 'window_6': 0.7795234987111022,
 'window_7': 0.8011349149082363,
 'window_8': 0.7844716887661726,
 'window_9': 0.7479558684011509,
 'window_10': 0.7826512340831657,
 'window_11': 0.8101035550520228}

In [10]:
values = list(metrics.values())
mean  = np.mean(values)
std = np.std(values)
print('mean and std: ',mean,std)

mean and std:  0.8063662616703037 0.0458301764245815


In [12]:
import pickle
outputfile = BASE_DIR/"results/backtest_wndow_expand.pkl"
with open(outputfile, 'wb') as f:
       pickle.dump({'metrics': metrics}, f)

print(f"Backtest complete. Metrics saved.")

Backtest complete. Metrics saved.


* the value of wrmsse mean and std, is fairly stable and does not require to do in depth postmortem of each window

In [14]:
from src.metrics import forecast_error


In [15]:

# # inspecting window 5 for bad wrmsse
# model_wn_2 = models['window_2']['point']
# end_date = win_end['window_2']
# pred_wn_2= predictions['window_2']

# start = train_data['date'].min()
# train_2,test_2 = split_data(train_data,start,end_date)

# fe = forecast_error(test_2,pred_wn_2)
# fe_desc = fe.sort_values()[::-1]
# selected = fe_desc.index.tolist()[:5]
# selected


In [16]:
# # 1. Merge test and prediction dataframes on unique identifiers
# df_2 = test_2.merge(pred_wn_2, on=['item_id', 'date'])

# # 2. Compute the in-band boolean flag across all rows
# df_2['in_band'] = (df_2['sales'] >= df_2['q10']) & (df_2['sales'] <= df_2['q90'])

# # 3. Calculate coverage per item_id
# item_coverage_2 = df_2.groupby('item_id')['in_band'].mean()

# # Optional: Calculate overall coverage across all items combined
# overall_coverage_2 = df_2['in_band'].mean()

# print('overall and item coverage for window 2',overall_coverage_2,item_coverage_2)

# items_below_  = item_coverage_2[item_coverage_2<0.6]
# items_below_

In [17]:
# # window : 9 , 
# model_wn_9 = models['window_9']
# end_date = win_end['window_9']
# pred_wn_9 = predictions['window_9']

# train_9,test_9  = split_data(train_data,start,end_date)
# fe_9 = forecast_error(test_9,pred_wn_9)
# selected_9 = fe_9.sort_values()[::-1].index.tolist()[:5]
# selected_9

In [18]:
# # 1. Merge test and prediction dataframes on unique identifiers
# df_9 = test_9.merge(pred_wn_9, on=['item_id', 'date'])

# # 2. Compute the in-band boolean flag across all rows
# df_9['in_band'] = (df_9['sales'] >= df_9['q10']) & (df_9['sales'] <= df_9['q90'])

# # 3. Calculate coverage per item_id
# item_coverage_9 = df_9.groupby('item_id')['in_band'].mean()

# # Optional: Calculate overall coverage across all items combined
# overall_coverage_9 = df_9['in_band'].mean()

# print('overall and item coverage foe window 9',overall_coverage_9,item_coverage_9)
# items_below_  = item_coverage_9[item_coverage_9<0.6]
# items_below_

In [19]:
def plot(raw_data, pred_data,items):
    fig, ax = plt.subplots(nrows=len(items),figsize=(15,10))

    for i, item in enumerate(items):
        raw = raw_data[raw_data['item_id']==item]
        pred = pred_data[pred_data['item_id']==item]

        ax[i].plot(raw['date'],raw['sales'],label='actual')
        ax[i].plot(pred['date'],pred['sales_pred'],label='predicted')
        ax[i].fill_between(pred['date'],pred['q10'],pred['q90'],alpha=0.2,color='green')
        
        ax[i].legend()
        ax[i].set_ylabel(item)
    plt.show()
    

In [20]:
# print('ploting for window_2')
# plot(test_2,pred_wn_2,selected)

In [21]:
# print('ploting for window_9')
# plot(test_9,pred_wn_9,selected_9)

In [22]:
# item = 'FOODS_3_090'

# plt.figure(figsize=(15,5))
# raw_train =  train_9[train_9['item_id']==item]
# plt.plot(raw_train['date'],raw_train['sales'])

In [23]:
# train_9 = build_train_features(train_9)

# item_raw = train_9[train_9['item_id']==item]

# item_raw_200 = item_raw[item_raw['sales']>200]
# # item_raw_200[['item_id','date','sales','snap_CA','event']]
# cols = [col for col in item_raw_200.columns if col.startswith('event_')]

# cols_selec = ['item_id','date','sales','sell_price','snap_CA','lag_7','lag_14','lag_28','rolling_mean_7','rolling_mean_28']+cols 



# item_raw_200[cols_selec]

**Of 373 items in the core holdout set, 350 (94%) achieve q10–q90 coverage within [60–100%], indicating well-calibrated quantile models. The remaining 23 items (6%) show coverage < 60%, suggesting demand patterns driven by factors outside the available feature set (e.g., undocumented promotions, wholesale orders)**

### Walk forward rolling window

In [ ]:
from src.backtests import walk_forward_rolling_window

In [13]:

# win_365, metrics_365, models_365, predictions_365= walk_forward_rolling_window(train_data)
# win_730,metrics_730,models_730,predictions_730 = walk_forward_rolling_window(train_data,training_window=730)
# win_1095,metrics_1095,models_1095,predictions_1095 = walk_forward_rolling_window(train_data,training_window=1095)

In [ ]:
# scores = list(metrics.values())
# std = np.std(scores)
# mean = np.mean(scores)
# print('mean and std over rolling window: ', mean, std)

mean and std over rolling window:  0.7972116753997809 0.0


In [ ]:
# scores_730 = list(metrics_730.values())
# std_730 = np.std(scores_730)
# mean_730 = np.mean(scores_730)
# print('mean and std over rolling window: ', mean_730, std_730)

mean and std over rolling window:  0.809182700826917 0.03648821202997754


**the best time period for our calculation is training on 2 years of data**

## Holding Cost for each item:

In [30]:
def cost_per_item(df_test,preds):
    '''pred: must contain point forecast and quantile forecasts'''

    item_data = df_test[['item_id','date','sell_price','sales']].copy()

    item_data = item_data.merge(preds,on=['item_id','date'],how='left')

    item_data['cost'] = 0.60*item_data['sell_price']
    item_data['profit']  = 0.40*item_data['sell_price']
    item_data['holding_cost'] = (0.25/365)*item_data['cost']
    item_data['stockout_cost'] = item_data['profit']

    item_data['safety_stock'] = item_data['q90'] - item_data['sales_pred']

    item_data['cost_holding'] = item_data['holding_cost']*item_data['safety_stock']
    item_data['expected_shortage'] = (item_data['sales'] - item_data['q90']).clip(lower=0)

    item_data['stockout_cost'] = item_data['stockout_cost']*item_data['expected_shortage']
    item_data['total_cost'] = item_data['cost_holding']  + item_data['stockout_cost']

    return item_data.groupby('item_id')['total_cost'].sum().reset_index()

    safety_stock = Q90 - point_forecast
    cost_holding_28d = holding_cost_daily * safety_stock * 28
    cost_stockout_28d = stockout_cost_per_unit * expected_shortage_28d

    total_cost = cost_holding_28d + cost_stockout_28d

In [27]:
from train_and_eval import *

train_data = sales_ca1_complete.copy()
cutoff = train_data['date'].max()-pd.Timedelta(days=28)

train = train_data[train_data['date']<=cutoff]
test  = train_data[train_data['date']>cutoff]

train_df = build_train_features(train)
models, cat_categories, params = train_models(train_df)

pred_df = predict_eval_set(models,train_df,test,cat_categories)

print(f"computing WRMSSE")
score = wrmsse(train_df,test,pred_df)
print(f" WRMSSE: {score:.4f}")

KeyboardInterrupt: 

In [35]:
items_with_costs  = cost_per_item(test,pred_df)
items_with_costs

,item_id,total_cost
0,FOODS_1_018,1.143748
1,FOODS_1_024,3.188498
2,FOODS_1_032,0.984013
3,FOODS_1_044,0.090975
4,FOODS_1_045,9.135942
...,...,...
368,HOUSEHOLD_2_450,2.944733
369,HOUSEHOLD_2_465,8.872073
370,HOUSEHOLD_2_483,0.118777
371,HOUSEHOLD_2_509,0.485338


In [36]:
test[['item_id','sales','sell_price']]

,item_id,sales,sell_price
27641,FOODS_1_018,7,0.979980
27642,FOODS_1_018,2,0.979980
27643,FOODS_1_018,2,0.979980
27644,FOODS_1_018,5,0.979980
27645,FOODS_1_018,6,0.979980
...,...,...,...
4365013,HOUSEHOLD_2_514,1,19.546875
4365014,HOUSEHOLD_2_514,0,19.546875
4365015,HOUSEHOLD_2_514,1,19.546875
4365016,HOUSEHOLD_2_514,0,19.546875


## benchmarking with a baseline: seasonal naive

In [37]:
# Create baseline predictions: sales from 28 days ago
train_sorted = train.sort_values(['item_id','date'])
pred_baseline = train_sorted[['item_id','date','sales']].copy()
pred_baseline['date'] = pred_baseline['date'] + pd.Timedelta(days=28)
pred_baseline = pred_baseline.rename(columns={'sales': 'sales_pred'})
pred_baseline['q90'] = pred_baseline['sales_pred']  # no safety margin
pred_baseline['q10'] = pred_baseline['sales_pred']

# Filter to only match test dates
pred_baseline = pred_baseline[pred_baseline['date'].isin(test['date'])]


score = wrmsse(train_sorted,test,pred_baseline)
print(f"wrmsse baseline: {score}")


# Now use your existing function
cost_baseline = cost_per_item(test, pred_baseline)
cost_optimized = cost_per_item(test, pred_df)  # your quantile preds

# Compare
comparison = cost_optimized.merge(cost_baseline, on='item_id', suffixes=('_optimized','_baseline'))
comparison['savings_pct'] = ((comparison['total_cost_baseline'] - comparison['total_cost_optimized']) / comparison['total_cost_baseline'] * 100).round(2)

total_savings_pct = ((comparison['total_cost_baseline'].sum() - comparison['total_cost_optimized'].sum()) / comparison['total_cost_baseline'].sum() * 100)
print(f"Overall savings with lgb model compared to baseline: {total_savings_pct:.1f}%")

wrmsse baseline: 1.0929312615801767
Overall savings with lgb model compared to baseline: 79.7%


# Final Model and stress testing on unknown future data:
* From the rolling window walk forward validation set, we observed that a training period of 2 years have better score 
* Lower variance — more stable across different time periods (std 0.036)
* Better average — mean 0.809 was the best

In [10]:
# reduce the data to recent two years
from train_and_eval import *

train_data = sales_ca1_complete.copy()

two_year_cutoff = train_data['date'].max()-pd.Timedelta(days=730)
train_data_new = train_data[train_data['date']>=two_year_cutoff]

min_date = train_data_new['date'].min()
max_date = train_data_new['date'].max()

print("after cutoff: ",min_date,max_date,(max_date-min_date).days, ' days')

after cutoff:  2014-01-05 00:00:00 2016-01-05 00:00:00 730  days


In [12]:
cutoff = train_data_new['date'].max()-pd.Timedelta(days=28)

train_new = train_data_new[train_data_new['date']<=cutoff]
test_new  = train_data_new[train_data_new['date']>cutoff]

train_new_df = build_train_features(train_new)
models, cat_categories, params = train_models(train_new_df)

pred_df= predict_eval_set(models,train_new_df,test_new,cat_categories)

print(f"computing WRMSSE")
score = wrmsse(train_new_df,test_new,pred_df)
print(f" WRMSSE with two year training: {score:.4f}")


computing WRMSSE
 WRMSSE with two year training: 0.8655


In [13]:
unique_items_ = train_data_new['item_id'].unique()

future_ca1_unique_items = sales_future_ca1[sales_future_ca1['item_id'].isin(unique_items_)].copy()

print("before filter: ",sales_future_ca1['item_id'].unique())
print("after filter: ",future_ca1_unique_items['item_id'].unique())

before filter:  ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', 'FOODS_1_005', ..., 'HOUSEHOLD_2_512', 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516']
Length: 3049
Categories (3049, object): ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', ..., 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516']
after filter:  ['FOODS_1_018', 'FOODS_1_024', 'FOODS_1_032', 'FOODS_1_044', 'FOODS_1_045', ..., 'HOUSEHOLD_2_450', 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']
Length: 373
Categories (3049, object): ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', ..., 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516']


In [14]:
# now retrain on full 2 year and apply model on unseen data

train_full = train_data_new.copy()
train_full_features = build_train_features(train_full)
model_2_year, cat_categories, params = train_models(train_full_features)


In [15]:
model_2_year

{'point': LGBMRegressor(learning_rate=0.05, max_depth=8, n_estimators=200,
               objective='tweedie', verbose=-1),
 'q10': LGBMRegressor(alpha=0.1, learning_rate=0.05, max_depth=8, n_estimators=200,
               objective='quantile', verbose=-1),
 'q90': LGBMRegressor(alpha=0.9, learning_rate=0.05, max_depth=8, n_estimators=200,
               objective='quantile', verbose=-1)}

In [19]:
import pickle

model_dir = BASE_DIR/'models'

point_model = model_2_year['point']
model_q10  = model_2_year['q10']
model_q90  = model_2_year['q90']

#unique items in the model
unique_items = train_data_new['item_id'].unique().tolist()
filename_output = model_dir / 'items_full_history_ca1.pkl'

with open(filename_output, 'wb') as f:
       pickle.dump({'items': unique_items}, f)

print(f"saved items name with full history")

joblib.dump(point_model,model_dir/'lgb_ca1_2yr_point.pkl')
joblib.dump(model_q10,model_dir/'lgb_ca1_2yr_q10.pkl')
joblib.dump(model_q90, model_dir/'lgb_ca1_2yr_q90.pkl')


# save this model for future

saved items name with full history


['/media/ashfaque/datas/ML-projects/retail-forecast-system/models/lgb_ca1_2yr_q90.pkl']

In [53]:
train_full_features['date']

26938     2014-01-05
26939     2014-01-06
26940     2014-01-07
26941     2014-01-08
26942     2014-01-09
             ...    
4365013   2016-01-01
4365014   2016-01-02
4365015   2016-01-03
4365016   2016-01-04
4365017   2016-01-05
Name: date, Length: 272663, dtype: datetime64[ns]

## checking for model stability:
To actually check for the model stability, we have to calculate the wrmsse score over different 28 period windows in the future 137 days

In [16]:
future_total_days = (future_ca1_unique_items['date'].max()-future_ca1_unique_items['date'].min()).days
num_windows = int(future_total_days/28)

future_start = future_ca1_unique_items['date'].min()

future_sale = future_ca1_unique_items.copy()


window_scores = {}

training_data = train_full_features.copy()
fixed_training   = training_data.copy()

for i in range(num_windows+1):
    future_end = future_start + pd.Timedelta(days=28)

    if future_end  > future_sale['date'].max():
        future_end = future_sale['date'].max()

    mask = (future_sale['date']>=future_start) & (future_sale['date']< future_end)
    future_window = future_sale[mask]


    print(f"window from: {future_start}-{future_end}: {(future_end-future_start).days} days")
    pred_wndw= predict_eval_set(model_2_year,training_data,future_window,cat_categories)

   
    preds = pred_wndw[['item_id','date','sales_pred']].copy().rename(columns={"sales_pred":'sales'})

    training_data = pd.concat((training_data[['item_id','date','sales']],preds))


    score_wndw = wrmsse(fixed_training,future_window,pred_wndw)
    print(f"score for the window {i} : ",score_wndw)
    window_scores[f'window_{i}'] = score_wndw

    print('training data shape', training_data.shape, '\n training data dates: ',training_data['date'].unique().tolist()[-1])

    future_start =future_end

    


window from: 2016-01-06 00:00:00-2016-02-03 00:00:00: 28 days
score for the window 0 :  0.7528060478676313
training data shape (283107, 3) 
 training data dates:  2016-02-02 00:00:00
window from: 2016-02-03 00:00:00-2016-03-02 00:00:00: 28 days
score for the window 1 :  0.8040666537874086
training data shape (293551, 3) 
 training data dates:  2016-03-01 00:00:00
window from: 2016-03-02 00:00:00-2016-03-30 00:00:00: 28 days
score for the window 2 :  0.8035458619281157
training data shape (303995, 3) 
 training data dates:  2016-03-29 00:00:00
window from: 2016-03-30 00:00:00-2016-04-27 00:00:00: 28 days
score for the window 3 :  0.7936692472078208
training data shape (314439, 3) 
 training data dates:  2016-04-26 00:00:00
window from: 2016-04-27 00:00:00-2016-05-22 00:00:00: 25 days
score for the window 4 :  0.8317588564156941
training data shape (323764, 3) 
 training data dates:  2016-05-21 00:00:00


In [44]:
window_scores

{'window_0': 0.7528060478676313,
 'window_1': 0.8040666537874086,
 'window_2': 0.8035458619281157,
 'window_3': 0.7936692472078208,
 'window_4': 0.8317588564156941}

In [19]:
max_date = train_full_features['date'].max()
print(max_date)

2016-01-05 00:00:00


In [20]:
train_for_lags_02 = train_full_features[['item_id','date','sales']].tail(100)
print("Notebook 02 - Last 100 training rows:")
print(train_for_lags_02)

Notebook 02 - Last 100 training rows:
                 item_id       date  sales
4364918  HOUSEHOLD_2_514 2015-09-28      0
4364919  HOUSEHOLD_2_514 2015-09-29      1
4364920  HOUSEHOLD_2_514 2015-09-30      0
4364921  HOUSEHOLD_2_514 2015-10-01      0
4364922  HOUSEHOLD_2_514 2015-10-02      0
...                  ...        ...    ...
4365013  HOUSEHOLD_2_514 2016-01-01      1
4365014  HOUSEHOLD_2_514 2016-01-02      0
4365015  HOUSEHOLD_2_514 2016-01-03      1
4365016  HOUSEHOLD_2_514 2016-01-04      0
4365017  HOUSEHOLD_2_514 2016-01-05      0

[100 rows x 3 columns]


In [21]:
train_full_features['item_id'].unique()

['FOODS_1_018', 'FOODS_1_024', 'FOODS_1_032', 'FOODS_1_044', 'FOODS_1_045', ..., 'HOUSEHOLD_2_450', 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']
Length: 373
Categories (610, object): ['FOODS_1_004', 'FOODS_1_012', 'FOODS_1_018', 'FOODS_1_023', ..., 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']

In [28]:
unique_items = pd.read_pickle(BASE_DIR/'models/items_full_history_ca1.pkl')['items']
# print(len(unique_items))
print(len(unique_items))
unique_items

373


['FOODS_1_018',
 'FOODS_1_024',
 'FOODS_1_032',
 'FOODS_1_044',
 'FOODS_1_045',
 'FOODS_1_046',
 'FOODS_1_055',
 'FOODS_1_067',
 'FOODS_1_080',
 'FOODS_1_085',
 'FOODS_1_086',
 'FOODS_1_088',
 'FOODS_1_103',
 'FOODS_1_104',
 'FOODS_1_137',
 'FOODS_1_144',
 'FOODS_1_150',
 'FOODS_1_152',
 'FOODS_1_161',
 'FOODS_1_173',
 'FOODS_1_177',
 'FOODS_1_178',
 'FOODS_1_180',
 'FOODS_1_183',
 'FOODS_1_187',
 'FOODS_1_194',
 'FOODS_1_197',
 'FOODS_1_206',
 'FOODS_1_217',
 'FOODS_1_218',
 'FOODS_1_219',
 'FOODS_2_013',
 'FOODS_2_014',
 'FOODS_2_019',
 'FOODS_2_021',
 'FOODS_2_029',
 'FOODS_2_030',
 'FOODS_2_047',
 'FOODS_2_052',
 'FOODS_2_054',
 'FOODS_2_056',
 'FOODS_2_059',
 'FOODS_2_060',
 'FOODS_2_063',
 'FOODS_2_082',
 'FOODS_2_101',
 'FOODS_2_103',
 'FOODS_2_104',
 'FOODS_2_116',
 'FOODS_2_128',
 'FOODS_2_139',
 'FOODS_2_141',
 'FOODS_2_150',
 'FOODS_2_153',
 'FOODS_2_164',
 'FOODS_2_173',
 'FOODS_2_181',
 'FOODS_2_183',
 'FOODS_2_197',
 'FOODS_2_198',
 'FOODS_2_212',
 'FOODS_2_233',
 'FOODS_

In [30]:
train_full_features.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,sell_price,day_of_week,day_of_month,week_of_year,revenue,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28
26938,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_1073,7,2014-01-05,11350,...,0.97998,6,5,1,6.859863,NaN,NaN,NaN,NaN,NaN
26939,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_1074,8,2014-01-06,11350,...,0.97998,0,6,2,7.839844,NaN,NaN,NaN,NaN,NaN
26940,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_1075,7,2014-01-07,11350,...,0.97998,1,7,2,6.859863,NaN,NaN,NaN,NaN,NaN
26941,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_1076,8,2014-01-08,11350,...,0.97998,2,8,2,7.839844,NaN,NaN,NaN,NaN,NaN
26942,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_1077,12,2014-01-09,11350,...,0.97998,3,9,2,11.759766,NaN,NaN,NaN,NaN,NaN


In [34]:
train_full_features.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'day_of_week',
       'day_of_month', 'week_of_year', 'revenue', 'lag_7', 'lag_14', 'lag_28',
       'rolling_mean_7', 'rolling_mean_28'],
      dtype='object')

In [32]:
known_data = pd.read_parquet(BASE_DIR/'data/processed/sales_known_ca_1.parquet')


#future data with only selected items
training_data_load = known_data[known_data['item_id'].isin(unique_items)]

training_data_load.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day_of_week,day_of_month,week_of_year
25866,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_1,17,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,1.0,5,29,4
25867,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_2,3,2011-01-30,11101,...,NaN,NaN,NaN,0,0,0,1.0,6,30,4
25868,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_3,8,2011-01-31,11101,...,NaN,NaN,NaN,0,0,0,1.0,0,31,5
25869,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_4,2,2011-02-01,11101,...,NaN,NaN,NaN,1,1,0,1.0,1,1,5
25870,FOODS_1_018_CA_1_evaluation,FOODS_1_018,FOODS_1,FOODS,CA_1,CA,d_5,5,2011-02-02,11101,...,NaN,NaN,NaN,1,0,1,1.0,2,2,5


In [55]:
training_data_load.shape

(672519, 25)

In [35]:
training_data_load.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'day_of_week',
       'day_of_month', 'week_of_year'],
      dtype='object')

In [58]:


future_total_days = (future_ca1_unique_items['date'].max()-future_ca1_unique_items['date'].min()).days
num_windows = int(future_total_days/28)

future_start = future_ca1_unique_items['date'].min()

future_sale = future_ca1_unique_items.copy()


window_scores = {}
cutoff = training_data_load['date'].max()-pd.Timedelta(days=730)
training_cut = training_data_load[training_data_load['date']>=cutoff]

training_feat = build_train_features(training_cut)

training_data = training_feat.copy()
fixed_training   = training_feat.copy()

for i in range(num_windows+1):
    future_end = future_start + pd.Timedelta(days=28)

    if future_end  > future_sale['date'].max():
        future_end = future_sale['date'].max()

    mask = (future_sale['date']>=future_start) & (future_sale['date']< future_end)
    future_window = future_sale[mask]


    print(f"window from: {future_start}-{future_end}: {(future_end-future_start).days} days")
    pred_wndw= predict_eval_set(model_2_year,training_data,future_window,cat_categories)

   
    preds = pred_wndw[['item_id','date','sales_pred']].copy().rename(columns={"sales_pred":'sales'})

    training_data = pd.concat((training_data[['item_id','date','sales']],preds))


    score_wndw = wrmsse(fixed_training,future_window,pred_wndw)
    print(f"score for the window {i} : ",score_wndw)
    window_scores[f'window_{i}'] = score_wndw

    print('training data shape', training_data.shape, '\n training data dates: ',training_data['date'].unique().tolist()[-1])

    future_start =future_end


window from: 2016-01-06 00:00:00-2016-02-03 00:00:00: 28 days
score for the window 0 :  0.7528060478676313
training data shape (283107, 3) 
 training data dates:  2016-02-02 00:00:00
window from: 2016-02-03 00:00:00-2016-03-02 00:00:00: 28 days
score for the window 1 :  0.8040666537874086
training data shape (293551, 3) 
 training data dates:  2016-03-01 00:00:00
window from: 2016-03-02 00:00:00-2016-03-30 00:00:00: 28 days
score for the window 2 :  0.8035458619281157
training data shape (303995, 3) 
 training data dates:  2016-03-29 00:00:00
window from: 2016-03-30 00:00:00-2016-04-27 00:00:00: 28 days
score for the window 3 :  0.7936692472078208
training data shape (314439, 3) 
 training data dates:  2016-04-26 00:00:00
window from: 2016-04-27 00:00:00-2016-05-22 00:00:00: 25 days
score for the window 4 :  0.8317588564156941
training data shape (323764, 3) 
 training data dates:  2016-05-21 00:00:00
